### This file is merely used to test the vector store retrieval capability.

In [5]:
SINGLE = True # Change to True if you want to use single chroma database for all documents
collection_name = "academic_documents" if not SINGLE else "vaa_documents"

from langchain.vectorstores import Chroma
from chromadb.config import Settings
from chromadb import Client, PersistentClient
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document

# Now using nomic-embed-text-v2-moe
embedding_function = OllamaEmbeddings(model="bge-m3:567m") # Please OPEN Ollama first!!

query = "How can I apply for a residential hall at the PolyU?"
embedding = embedding_function.embed_query(query)

In [6]:
client = Client(Settings())
client = PersistentClient(path="../chroma_db")
collection = client.get_collection(name=collection_name)

vectorStore = Chroma(
    collection_name=collection_name, 
    client=client, 
    embedding_function=embedding_function)

collections = client.get_collection(collection_name)
docs = collections.get(
    include=["documents", "metadatas"],
    limit=collections.count()
)

docs_for_bm25 = [
    Document(page_content=doc_text, metadata=md)
    for doc_text, md in zip(docs["documents"], docs["metadatas"])
]

bm25_retriever = BM25Retriever.from_documents(docs_for_bm25)
print(f"BM25 retriever: initialized over {len(docs_for_bm25)} documents")

BM25 retriever: initialized over 15867 documents


In [7]:
docs = vectorStore.similarity_search(query, k=20)

for doc in docs:
    '''
    if "original_table" in doc.metadata:
        print("[Swapping for Raw Markdown Table]")
        new_result = doc.metadata["original_table"]
    else:
        new_result = doc.page_content
    '''
    
    print("========================================================")
    print(f"Content: {doc.page_content}...")
    print(f"Source: {doc.metadata.get('source')}")
    print(f"Chunk ID: {doc.metadata.get('chunk_id')}\n")

Content: --- PolyU Student Affairs Office (SAO) Website ---

Students are strongly encouraged to have hall-life experience during their university years, which is both memorable and rewarding. The Student Halls of Residence do not just provide students with an accommodation, but a vibrant community with abundant opportunities for them to grow and learn.
Application for Hall Residence 2025/26 – Full-time undergraduate students
Student Type
Application Period
Ad-hoc Application for Hall Residence 2025/26
Stage (1): 27 Jan (10:00am) – 2 Feb 2026 (11:59pm)
Stage (2): 3 Feb (10:00am) - 24 Mar 2026 (11:59pm)
Inbound exchange students of Semester 2*
2 Dec 2025 (10:00am) - 9 Dec 2025 (11:59pm)
*Global Engagement Office will inform eligible inbound exchange students the application and arrangement of hall accommodation by email in due course.
Application for Summer Hall Residence 2026 – Full-time undergraduate students
Student Type
Application Period
Full-time undergraduate students
13 Apr (10:

In [8]:
bm25_docs = bm25_retriever.get_relevant_documents(query)[:5]

for doc in bm25_docs:
    '''
    if "original_table" in doc.metadata:
        print("[Swapping for Raw Markdown Table]")
        new_result = doc.metadata["original_table"]
    else:
        new_result = doc.page_content
    '''
    
    print("========================================================")
    print(f"Content: {doc.page_content}...")
    print(f"Source: {doc.metadata.get('source')}")
    print(f"Chunk ID: {doc.metadata.get('chunk_id')}\n")

Content: --- PolyU Student Affairs Office (SAO) Website ---

6. After check-out, students will not be able to enter the hall and their room.
Please note that merely vacating the room and moving-out of personal properties without going through the official check-out procedures will NOT be regarded as check-out.
Am I eligible for a refund if I check out early?
Residents are not entitled to any adjustment on Hall charges for late occupancy, or for early withdrawal during the last two months of the residential year, as the residential places have been reserved and accepted.
Cancellation and Withdrawal of Hall Residence
a.
If a resident withdraws his/her hall residence prior to check-in or before the last 2 months of the residential year, a Residence Cancellation Fee will be levied as the administration charge. The hall lodging fee settled will be refunded. If a resident decides not to check in but fails to process his/her withdrawal before his/her expected check-in date, on top of a Reside